**Step 1: Generate Synthetic Data**                                         
I created a normal signal (like server CPU usage) using a sine wave with some  noise, and  then inject sudden "spikes" that represent *the incidents*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
time_steps = 10000
time = np.arange(time_steps)
cpu_usage = np.sin(time / 10) + np.random.normal(0, 0.5, time_steps)

incident_indices = np.random.choice(time_steps, size=50, replace=False)
for idx in incident_indices:
    cpu_usage[idx:idx+5] += np.random.uniform(5, 10)

df = pd.DataFrame({'time': time, 'cpu_usage': cpu_usage, 'is_incident': 0})
for idx in incident_indices:
    df.loc[idx:idx+5, 'is_incident'] = 1

**Step 2: The Sliding Window Formulation**                                     
You need to predict if an incident happens in the next $H$ steps based on the past $W$ steps. We must transform our 1D time-series into a 2D tabular format (rows = windows, columns = past steps).

In [ ]:
def create_sliding_windows(data, target, W, H):
    X, y = [], []
    for i in range(len(data) - W - H + 1):
        window_features = data[i : i + W]
        future_target = target[i + W : i + W + H]
        incident_in_future = 1 if sum(future_target) > 0 else 0
        X.append(window_features)
        y.append(incident_in_future)
    return np.array(X), np.array(y)

W = 60
H = 10
X, y = create_sliding_windows(df['cpu_usage'].values, df['is_incident'].values, W, H)

**Step 3: Train-Test Split**


In [ ]:
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

**Step 4: Model Selection and Training**                                             
Since our sliding window transformed the data into standard tabular rows, we can use standard models. *Random Forest* is an excellent choice here—it requires very little tuning and handles tabular feature relationships perfectly.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

**Step 5: Evaluation and Alert Thresholds**

In [ ]:
from sklearn.metrics import classification_report, precision_recall_curve
import seaborn as sns

y_probs = model.predict_proba(X_test)[:, 1]
ALERT_THRESHOLD = 0.70
y_pred_custom = (y_probs >= ALERT_THRESHOLD).astype(int)

print("Evaluation with Alert Threshold =", ALERT_THRESHOLD)
print(classification_report(y_test, y_pred_custom))